In [ ]:
!pip install "unstructured[pdf]" pytesseract pillow pdfplumber

In [ ]:
from unstructured.partition.pdf import partition_pdf

elements = partition_pdf(
    filename="sample.pdf",
    strategy="hi_res",
    infer_table_structure=True
)

print(f"Total Elements: {len(elements)}\n")

for i, element in enumerate(elements):
    print("=" * 80)
    print(f"Element {i+1}")
    print(f"Type: {type(element).__name__}")
    print()

    try:
        print(element.text[:1000])
    except:
        print("No text available")

    print("\n")

In [ ]:
from unstructured.partition.pdf import partition_pdf
from unstructured.documents.elements import Table

elements = partition_pdf(
    filename="sample.pdf",
    strategy="hi_res",
    infer_table_structure=True
)

for element in elements:
    if isinstance(element, Table):

        print("="*100)

        if hasattr(element.metadata, "text_as_html"):
            print(element.metadata.text_as_html)

        break

In [ ]:
!pip install pymupdf

In [ ]:
import fitz  # PyMuPDF
import time

def extract_with_pymupdf(pdf_path):
    start = time.time()

    doc = fitz.open(pdf_path)
    all_text = []

    for page_num, page in enumerate(doc):
        text = page.get_text("text")  # plain text extraction
        all_text.append({
            "page": page_num + 1,
            "text": text
        })

    doc.close()

    end = time.time()

    return all_text, end - start


if __name__ == "__main__":
    pdf_path = "sample.pdf"

    data, duration = extract_with_pymupdf(pdf_path)

    print(f"Time taken: {duration:.2f} seconds")
    print("\n--- PAGE SAMPLE ---\n")
    print(data[0]["text"][:1000])  # first page preview

In [ ]:
import pdfplumber
import time

def extract_with_pdfplumber(pdf_path):
    start = time.time()

    results = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):

            text = page.extract_text()

            tables = page.extract_tables()

            results.append({
                "page": page_num + 1,
                "text": text,
                "tables": tables
            })

    end = time.time()

    return results, end - start


if __name__ == "__main__":
    pdf_path = "sample.pdf"

    data, duration = extract_with_pdfplumber(pdf_path)

    print(f"Time taken: {duration:.2f} seconds")

    print("\n--- PAGE 1 TEXT ---\n")
    print(data[0]["text"][:1000])

    print("\n--- PAGE 1 TABLES ---\n")
    for i, table in enumerate(data[0]["tables"]):
        print(f"\nTable {i+1}:")
        for row in table:
            print(row)

In [ ]:
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community

In [ ]:
from unstructured.partition.pdf import partition_pdf
import os

# Ensure the output directory exists
output_path = "./content/"
os.makedirs(output_path, exist_ok=True)

# Fixed the double .pdf extension typo
file_path = '/content/sample.pdf'

chunks = partition_pdf(
    filename=file_path,
    infer_table_structure=True,
    strategy="hi_res",

    chunking_strategy="by_title",
    max_characters=10000,
    combine_text_under_n_chars=2000,
    new_after_n_chars=6000,
)

In [ ]:
elements = chunks

In [ ]:
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")


In [ ]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],

        'types': ['text']
    }

    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            # Handle images


    content_data['types'] = list(set(content_data['types']))
    return content_data


In [ ]:
!pip install -qU langchain-groq

In [ ]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document

from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage

In [ ]:

from langchain_groq import ChatGroq
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""

    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)

        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """

        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add images to the message

        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content

    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [ ]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")

        # Analyze chunk content
        content_data = separate_content_types(chunk)

        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}")

        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']

        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                })
            }
        )

        langchain_documents.append(doc)

    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents


# Process chunks with AI
processed_chunks = summarise_chunks(elements)

In [ ]:
processed_chunks

In [ ]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []

    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)

    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

In [ ]:
!pip install langchain_huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
    """Create and persist ChromaDB vector store"""
    print("🔮 Creating embeddings and storing in ChromaDB...")

    embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
    )


    # Create ChromaDB vector store
    print("--- Creating vector store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    print("--- Finished creating vector store ---")

    print(f"✅ Vector store created and saved to {persist_directory}")
    return vectorstore

# Create the vector store
db = create_vector_store(processed_chunks)

In [ ]:
# After your retrieval
query = "what is location and the date"
retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

# Export to JSON
export_chunks_to_json(chunks, "rag_results.json")

In [ ]:
query = "what are the attendees"

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""

    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatGroq(model="qwen/qwen3-32b", temperature=0)

        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""

        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"

            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])

                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"

                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"

            prompt_text += "\n"

        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]

        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])




        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])

        return response.content

    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)